# Symbolic Differentiation

In this notebook our goal is to implement *symbolic differentiation*.  Concretely, we want to implement a function `diff` that takes one argument:
  - The argument `expr` represents an *arithmetic expression*.
    Here an arithmetic expression is any string that is build from variable and numbers by application
    of any of the operator symbols "`+`", "`-`", "`*`", "`/`", and "`**`".
    The operator "`**`" represents exponentiation, i.e. an expression of the form
    $a \texttt{**} b$ is interpreted as $a^b$.
  - If $e$ is an expression, then both $\exp(e)$ and $\ln(e)$ are expressions too.

The function call `diff expr` will then compute the derivative of `expr` with respect to the variable `x`.  For example, the function call
`diff "x * exp(x)"` will compute the output
`1 * exp(x) + x * exp(x)` because we have:
$$ \frac{\mathrm{d}\;}{\mathrm{d}x} \bigl( x \cdot \mathrm{e}^x \bigr) = 1 \cdot \mathrm{e}^x + x \cdot \mathrm{e}^x. $$

This file shows the implementation of a program that can perform *symbolic differentiation* using the scanner
generator `ocamllex` and the parser generator `Menhir`.  (The Python version uses the library `lark`.)
We will convert the parse tree into an abstract syntax tree so that our symbolic differentiation function can leverage OCaml's pattern matching.

## Specification of the Grammar and Parser

The rules of the grammar inherently enforce mathematical precedence. We include a rule `unary` to handle unary minus correctly relative to exponentiation and multiplication.
As explained in the notebook `01-Calculator.ipynb`, the scanner and the grammar are specified separately, and the
semantic actions of the grammar build a parse tree in the style of `lark`.  The keywords `exp` and `ln` and the
variable `x` are tokens of their own.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree

In [ ]:
let calc_lexer = {|
{
  open Diff_parser
}

let digit  = ['0'-'9']
let int    = digit+
let number = int ['e' 'E'] ['+' '-']? int
           | (int '.' int? | '.' int) (['e' 'E'] ['+' '-']? int)?
           | int

rule token = parse
  | [' ' '\t' '\r' '\n' '\012']+ { token lexbuf }                (* ignore whitespace *)
  | "**"                         { POW }
  | '+'                          { PLUS }
  | '-'                          { MINUS }
  | '*'                          { STAR }
  | '/'                          { SLASH }
  | '('                          { LPAR }
  | ')'                          { RPAR }
  | "exp"                        { EXP }
  | "ln"                         { LN }
  | 'x'                          { X }
  | number                       { NUMBER (Parse_tree.token "NUMBER" lexbuf) }
  | eof                          { EOF }
  | _                            { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ Lexing.lexeme lexbuf)) }
|}

let calc_grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> NUMBER
%token PLUS MINUS STAR SLASH POW LPAR RPAR EXP LN X EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> expr product unary factor base

%%

start: e = expr EOF                         { e }

expr: e = expr PLUS p = product             { Tree ("add", [e; p]) }
    | e = expr MINUS p = product            { Tree ("sub", [e; p]) }
    | p = product                           { p }

product: p = product STAR u = unary         { Tree ("mul", [p; u]) }
       | p = product SLASH u = unary        { Tree ("div", [p; u]) }
       | u = unary                          { u }

unary: MINUS u = unary                      { Tree ("neg", [u]) }
     | f = factor                           { f }

factor: b = base POW u = unary              { Tree ("pow", [b; u]) }
      | b = base                            { b }

base: EXP LPAR e = expr RPAR                { Tree ("exp", [e]) }
    | LN LPAR e = expr RPAR                 { Tree ("ln", [e]) }
    | LPAR e = expr RPAR                    { e }
    | n = NUMBER                            { Tree ("number", [Token n]) }
    | X                                     { Tree ("var", []) }
|}

We generate a scanner and a parser for this grammar.

In [ ]:
generate_parser ~name:"diff" ~lexer:calc_lexer ~grammar:calc_grammar

## AST Construction

The functions below turn the parse tree into an abstract syntax tree of the type `expr`.  In Python, the abstract
syntax trees are nested tuples like `('+', ('ln', 'x'), 1)`.  In OCaml, every operator is a constructor of the type
`expr`, e.g. `Add (Ln X, Num (I 1))`.

Python distinguishes the numbers of type `int` from those of type `float`, and the simplification below depends on
this distinction: for example, `1 / 2` is the floating point number `0.5`, while `2 ** 3` is the integer `8`.  The
type `number` reproduces this: `I n` is an integer and `F x` is a floating point number.

The table `operatorTable` maps the names of the binary nodes of the parse tree to the constructors of the type `expr`.

In [ ]:
type number = I of int | F of float

type expr =
  | Num of number
  | X                         (* the variable x *)
  | Add of expr * expr        (* f + g  *)
  | Sub of expr * expr        (* f - g  *)
  | Mul of expr * expr        (* f * g  *)
  | Div of expr * expr        (* f / g  *)
  | Pow of expr * expr        (* f ** g *)
  | Neg of expr               (* -f     *)
  | Exp of expr               (* exp(f) *)
  | Ln  of expr               (* ln(f)  *)

let operatorTable : (string * (expr -> expr -> expr)) list =
  [ ("add", fun f g -> Add (f, g)); ("sub", fun f g -> Sub (f, g)); ("mul", fun f g -> Mul (f, g));
    ("div", fun f g -> Div (f, g)); ("pow", fun f g -> Pow (f, g)) ]

In [ ]:
let rec ast_to_tuples (node : tree) : expr =
  match node with
  | Tree ("number", [ Token t ]) -> Num (I (int_of_string t.value))
  | Tree ("var", []) -> X
  | Tree ("neg", [ f ]) -> Neg (ast_to_tuples f)
  | Tree ("exp", [ f ]) -> Exp (ast_to_tuples f)
  | Tree ("ln", [ f ]) -> Ln (ast_to_tuples f)
  | Tree (op_name, [ f; g ]) when List.mem_assoc op_name operatorTable ->
      (List.assoc op_name operatorTable) (ast_to_tuples f) (ast_to_tuples g)
  | _ -> invalid_arg ("ast_to_tuples: unexpected node " ^ Parse_tree.to_string node)

In [ ]:
let parse (s : string) : expr =
  let tree = parse_string Diff_parser.start Diff_lexer.token s in
  ast_to_tuples tree

In [ ]:
parse "ln(x ** x) + exp(x * -x)"

## Symbolic Differentiation

Now we are ready to implement the function `diff` that takes an expression possibly containing the variable `x`.
It computes the derivative of the given expression with respect to `x`.
<ol>
<li>  The first case implements the rule:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) + g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) + \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The second case implements the rule:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) - g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) - \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The third case deals with the product rule.
      </li>
<li>  The fourth case deals with the quotient rule.
      </li>
<li>  The fifth case deals with exponentiation trick (rewriting using ln and exp).
      </li>
<li>  The sixth case deals with unary negation:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(-f(x)\bigr) = -\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)$$
      </li>
<li>  The seventh case deals with natural logarithm.
      </li>
<li>  The eighth case deals with the exponential function.
      </li>
<li>  The derivative of `x` is 1.  Otherwise, the expression is a constant and hence we return 0.
      </li>
</ol>

Since the compiler checks that `diff` handles every constructor of the type `expr`, we can not forget a case.

In [ ]:
let rec diff (e : expr) : expr =
  (* differentiate the expression e with respect to the variable x *)
  match e with
  | Add (f, g) ->
      let fs, gs = diff f, diff g in
      Add (fs, gs)
  | Sub (f, g) ->
      let fs, gs = diff f, diff g in
      Sub (fs, gs)
  | Mul (f, g) ->
      let fs, gs = diff f, diff g in
      Add (Mul (fs, g), Mul (f, gs))
  | Div (f, g) ->
      let fs, gs = diff f, diff g in
      Div (Sub (Mul (fs, g), Mul (f, gs)), Mul (g, g))
  | Pow (f, g) -> diff (Exp (Mul (g, Ln f)))
  | Neg f -> Neg (diff f)
  | Ln f ->
      let fs = diff f in
      Div (fs, f)
  | Exp f ->
      let fs = diff f in
      Mul (fs, e)
  | X -> Num (I 1)
  | Num _ -> Num (I 0)

### Arithmetic with Numbers

The simplification rules below compute with numbers.  The following functions implement the arithmetic of Python:
- The result of `+`, `-`, and `*` is an integer if both arguments are integers and a floating point number otherwise.
- The result of `/` is always a floating point number.
- The result of `**` is an integer if both arguments are integers and the exponent is not negative.
- Integers and floating point numbers are compared by their values, e.g. `1 == 1.0` is true.  The function
  `expr_equal` compares expressions in the same way, like Python compares nested tuples.

In [ ]:
let to_float (n : number) : float = match n with I k -> float_of_int k | F x -> x

let num_eq (a : number) (b : number) : bool = to_float a = to_float b

let arith (int_op : int -> int -> int) (float_op : float -> float -> float) (a : number) (b : number) : number =
  match a, b with
  | I x, I y -> I (int_op x y)
  | _ -> F (float_op (to_float a) (to_float b))

let num_add = arith ( + ) ( +. )
let num_sub = arith ( - ) ( -. )
let num_mul = arith ( * ) ( *. )
let num_div (a : number) (b : number) : number = F (to_float a /. to_float b)

let num_pow (a : number) (b : number) : number =
  match a, b with
  | I x, I y when y >= 0 -> I (int_of_float (Float.pow (float_of_int x) (float_of_int y)))
  | _ -> F (Float.pow (to_float a) (to_float b))

let num_neg (a : number) : number = match a with I x -> I (-x) | F x -> F (-.x)

let num_to_string (a : number) : string = match a with I k -> string_of_int k | F x -> float_repr x

(* is_value k e checks whether e is the number k *)
let is_value (k : int) (e : expr) : bool = match e with Num n -> num_eq n (I k) | _ -> false

let rec expr_equal (e1 : expr) (e2 : expr) : bool =
  match e1, e2 with
  | Num a, Num b -> num_eq a b
  | X, X -> true
  | Add (a1, b1), Add (a2, b2) | Sub (a1, b1), Sub (a2, b2) | Mul (a1, b1), Mul (a2, b2)
  | Div (a1, b1), Div (a2, b2) | Pow (a1, b1), Pow (a2, b2) ->
      expr_equal a1 a2 && expr_equal b1 b2
  | Neg a1, Neg a2 | Exp a1, Exp a2 | Ln a1, Ln a2 -> expr_equal a1 a2
  | _ -> false

### Summary of Simplification Rules

The `simplify` function applies the following algebraic identities to transform expression trees into their canonical, reduced forms. Simplification is performed bottom-up, with recursive re-evaluation to ensure chained simplifications (e.g., `--a -> a`) are fully applied.

#### 1. Constant Folding
* All arithmetic operations (`+`, `-`, `*`, `/`, `**`) are evaluated whenever both operands are numeric constants.

#### 2. Unary Negation Folding
* $-(-a) \rightarrow a$
* $-0 \rightarrow 0$
* $-a$ (where $a$ is a constant) $\rightarrow$ numeric negation

#### 3. Addition & Subtraction Identities
* $a + 0 = a$
* $0 - a = -a$
* $a - 0 = a$
* $a - a = 0$
* $a + (-b) \rightarrow a - b$
* $-a + b \rightarrow b - a$
* $a - (-b) \rightarrow a + b$

#### 4. Multiplication & Reciprocal Rules
* $a \cdot 0 = 0$
* $a \cdot 1 = a$
* $-1 \cdot a \rightarrow -a$
* $a \cdot \frac{1}{a} = 1$
* $-a \cdot \frac{1}{a} = -1$
* $a \cdot (-\frac{1}{a}) = -1$

#### 5. Power Derivative Reduction
*Used to simplify chain rule outputs:*
* $a \cdot \frac{1}{f} \cdot f^b \rightarrow a \cdot f^{b-1}$
* $\frac{1}{f} \cdot f^b \rightarrow f^{b-1}$
* *(Special Case $b=2$)*: $a \cdot \frac{1}{f} \cdot f^2 \rightarrow a \cdot f$

#### 6. Exponentiation, Logarithmic & Exponential Identities
* $a^0 = 1$, $a^1 = a$, $0^a = 0$, $1^a = 1$
* $\ln(1) = 0$
* $\ln(\exp(a)) = a$
* $\exp(0) = 1$
* $\exp(\ln(a)) = a$
* $\exp(a \cdot \ln(f)) \rightarrow f^a$

As in Python, the cases are tried from top to bottom and the first case that matches is used.  A pattern of the
Python version that contains a number, e.g. `('+', 0, a)`, becomes a pattern with a condition, e.g.
`Add (z, a) when is_value 0 z`, and a condition like `if a == b` becomes `when expr_equal a b`.

In [ ]:
let rec simplify (e : expr) : expr =
  (* Recursively simplifies an abstract syntax tree representing an arithmetic expression. *)
  match e with
  | Num _ | X -> e
  | _ ->
      (* 1. Recursively simplify sub-expressions first (bottom-up) *)
      let e_sim =
        match e with
        | Add (l, r) -> Add (simplify l, simplify r)
        | Sub (l, r) -> Sub (simplify l, simplify r)
        | Mul (l, r) -> Mul (simplify l, simplify r)
        | Div (l, r) -> Div (simplify l, simplify r)
        | Pow (l, r) -> Pow (simplify l, simplify r)
        | Neg a -> Neg (simplify a)
        | Exp a -> Exp (simplify a)
        | Ln a -> Ln (simplify a)
        | Num _ | X -> e
      in
      (* 2. Apply algebraic simplification rules *)
      match e_sim with
      (* Unary Negation Folding *)
      | Neg (Num a) -> Num (num_neg a)
      | Neg (Neg a) -> a
      | Neg z when is_value 0 z -> Num (I 0)

      (* Constant Folding *)
      | Add (Num a, Num b) -> Num (num_add a b)
      | Sub (Num a, Num b) -> Num (num_sub a b)
      | Mul (Num a, Num b) -> Num (num_mul a b)
      | Div (Num a, Num b) -> if num_eq b (I 0) then e_sim else Num (num_div a b)
      | Pow (Num a, Num b) -> Num (num_pow a b)

      (* Addition & Subtraction *)
      | Add (z, a) when is_value 0 z -> a
      | Add (a, z) when is_value 0 z -> a
      | Add (a, Neg b) -> simplify (Sub (a, b))
      | Add (Neg a, b) -> simplify (Sub (b, a))

      (* Force re-evaluation on new structures *)
      | Sub (z, a) when is_value 0 z -> simplify (Neg a)
      | Sub (a, Neg b) -> simplify (Add (a, b))

      | Sub (a, z) when is_value 0 z -> a
      | Sub (a, b) when expr_equal a b -> Num (I 0)

      (* Multiplication *)
      | Mul (z, _) when is_value 0 z -> Num (I 0)
      | Mul (_, z) when is_value 0 z -> Num (I 0)
      | Mul (o, a) when is_value 1 o -> a
      | Mul (a, o) when is_value 1 o -> a

      (* Force re-evaluation on new structures *)
      | Mul (m, a) when is_value (-1) m -> simplify (Neg a)
      | Mul (a, m) when is_value (-1) m -> simplify (Neg a)

      | Mul (a, Div (o, b)) when is_value 1 o && expr_equal a b -> Num (I 1)
      | Mul (Div (o, b), a) when is_value 1 o && expr_equal a b -> Num (I 1)
      | Mul (Neg a, Div (o, b)) when is_value 1 o && expr_equal a b -> Num (I (-1))
      | Mul (Div (o, b), Neg a) when is_value 1 o && expr_equal a b -> Num (I (-1))
      | Mul (a, Neg (Div (o, b))) when is_value 1 o && expr_equal a b -> Num (I (-1))
      | Mul (Neg (Div (o, b)), a) when is_value 1 o && expr_equal a b -> Num (I (-1))

      (* Power Derivative Reduction Rules *)
      | Mul (Mul (a, Div (o, f)), Pow (f2, Num b)) when is_value 1 o && expr_equal f f2 ->
          if num_eq b (I 2) then Mul (a, f) else Mul (a, Pow (f, Num (num_sub b (I 1))))
      | Mul (Pow (f2, Num b), Mul (a, Div (o, f))) when is_value 1 o && expr_equal f f2 ->
          if num_eq b (I 2) then Mul (a, f) else Mul (a, Pow (f, Num (num_sub b (I 1))))
      | Mul (Div (o, f), Pow (f2, Num b)) when is_value 1 o && expr_equal f f2 ->
          if num_eq b (I 2) then f else Pow (f, Num (num_sub b (I 1)))
      | Mul (Pow (f2, Num b), Div (o, f)) when is_value 1 o && expr_equal f f2 ->
          if num_eq b (I 2) then f else Pow (f, Num (num_sub b (I 1)))

      (* Division *)
      | Div (z, _) when is_value 0 z -> Num (I 0)
      | Div (a, o) when is_value 1 o -> a
      | Div (a, b) when expr_equal a b -> Num (I 1)

      (* Exponentiation *)
      | Pow (_, z) when is_value 0 z -> Num (I 1)
      | Pow (a, o) when is_value 1 o -> a
      | Pow (z, _) when is_value 0 z -> Num (I 0)
      | Pow (o, _) when is_value 1 o -> Num (I 1)

      (* Logarithmic & Exponential Identities *)
      | Ln o when is_value 1 o -> Num (I 0)
      | Ln (Exp a) -> a
      | Exp z when is_value 0 z -> Num (I 1)
      | Exp (Ln a) -> a
      | Exp (Mul (a, Ln f)) -> Pow (f, a)
      | Exp (Mul (Ln f, a)) -> Pow (f, a)

      (* Fallback *)
      | _ -> e_sim

## String Formatting
The function `precedenceOp` ranks the operators, including the unary operator `Neg`.

In [ ]:
let precedenceOp (e : expr) : int =
  match e with
  | Add _ | Sub _ -> 1
  | Mul _ | Div _ -> 2
  | Neg _ -> 3
  | Pow _ -> 4
  | Num _ | X | Exp _ | Ln _ -> 5

let rec toString (e : expr) : string =
  let paren cond s = if cond then "(" ^ s ^ ")" else s in
  match e with
  | Num n -> num_to_string n
  | X -> "x"
  | Neg a ->
      (* Only wrap in parens if inner expr is lower precedence than mult/div (+ or -) *)
      if precedenceOp a <= 1 then "-(" ^ toString a ^ ")" else "-" ^ toString a
  | Exp a -> "exp(" ^ toString a ^ ")"
  | Ln a -> "ln(" ^ toString a ^ ")"
  | Add (l, r) -> toString l ^ " + " ^ toString r
  | Sub (l, r) -> toString l ^ " - " ^ paren (precedenceOp r = 1) (toString r)
  | Mul (l, r) -> paren (precedenceOp l = 1) (toString l) ^ "*" ^ paren (precedenceOp r = 1) (toString r)
  | Div (l, r) -> paren (precedenceOp l = 1) (toString l) ^ "/" ^ paren (precedenceOp r <= 2) (toString r)
  | Pow (l, r) -> paren (precedenceOp l <= 4) (toString l) ^ "**" ^ paren (precedenceOp r <= 3) (toString r)

## Testing

In [ ]:
let test (s : string) : unit =
  let t = parse s in
  let d = simplify (diff t) in
  Printf.printf "d/dx %s = %s\n%!" s (toString d)

In [ ]:
test "ln(x ** x)"

In [ ]:
test "x ** -x"

In [ ]:
test "-x ** 2"

In [ ]:
test "1/(1+exp(-x))"

In [ ]:
test "x*x"

In [ ]:
test "x/x"